## PURPOSE OF THIS NOTEBOOK: 

The goal is to create a pitstop calculator based on telemetry data. It will report: 
1. average fuel consumption 
2. fuel consumption needed to add a lap before pitstop
3. total time estimated from per lap fuel saved
4. laptime expected for level of fuel save

### Factors I would need to calculate to complete this 
1. rate of acceleration

### NOTES: 
1. Data was collected varying frequencies. Given we know the frequency cycle rates, the formula to conver to seconds will be (F = 1/T) 
2. Tables labeled as Events report the event in seconds 
3. I need to think of a way to calculate fuel used when additional fuel is put in during a pitstop. First I need to isolate the lap where
4. Cars of all classes in LMU all have 100L liters of fuel. From the limited testing I've done the fuel state reported in telemetry is the liters of fuel left, not virtual energy

## Step 01: Explore Data

In [1]:
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
# import the data we need 

# Lap, this has the time in seconds where the lap started as well as the lap number, will be used as a primary indicator for position 
lap = pd.read_csv("data/Lap.csv")

# Fuel Level, the fuel level needed, Frequency is 20
fuel_level = pd.read_csv("data/Fuel Level.csv")

In [3]:
fuel_level.shape

(50908, 1)

In [4]:
fuel_level

,value
0,92.000000
1,92.000000
2,92.000000
3,92.000000
4,92.000000
...,...
50903,16.962720
50904,16.962633
50905,16.962545
50906,16.962456


In [5]:
# Create new row, ts for fuel_level, will represent the amount of time in seconds. 
# we know the frequency is 20, which means we get a reading every .05 seconds (using the frequency equation)
fuel_level['ts'] = .05 * fuel_level.index

# renaming fuel level for merge
fuel_level.rename({"value" : "fuel_level"}, inplace=True, axis=1)

In [6]:
fuel_level

,fuel_level,ts
0,92.000000,0.00
1,92.000000,0.05
2,92.000000,0.10
3,92.000000,0.15
4,92.000000,0.20
...,...,...
50903,16.962720,2545.15
50904,16.962633,2545.20
50905,16.962545,2545.25
50906,16.962456,2545.30


In [7]:
lap

,ts,value
0,14.6425,0
1,238.5600,1
2,345.8400,2
3,452.9800,3
4,559.9200,4
5,665.9000,5
6,773.0600,6
7,879.4600,7
8,985.8000,8
9,1093.9200,9


### Goal: 

I want to add the lap numbers to the fuel level to know which lap is which.
I calculated the amount of time passed, now I need to make every row where the time is less than the lap assciocated with the other, give it the lap it's in

In [8]:
fuel_level

,fuel_level,ts
0,92.000000,0.00
1,92.000000,0.05
2,92.000000,0.10
3,92.000000,0.15
4,92.000000,0.20
...,...,...
50903,16.962720,2545.15
50904,16.962633,2545.20
50905,16.962545,2545.25
50906,16.962456,2545.30


In [9]:
fuel_level

,fuel_level,ts
0,92.000000,0.00
1,92.000000,0.05
2,92.000000,0.10
3,92.000000,0.15
4,92.000000,0.20
...,...,...
50903,16.962720,2545.15
50904,16.962633,2545.20
50905,16.962545,2545.25
50906,16.962456,2545.30


In [ ]:
# Create a new column, lap, based on the amount of time passed using the lap table as a LUT
# I want to 
def lookup_up_lap(times):
    time_index = times <= lap['ts']
    lap_num = lap['value'][time_index]
    return lap_num.values[0]

fuel_level['LapNum'] = fuel_level['ts'].apply(lookup_up_lap)

In [11]:
fuel_level

,fuel_level,ts,LapNum
0,92.000000,0.00,0
1,92.000000,0.05,0
2,92.000000,0.10,0
3,92.000000,0.15,0
4,92.000000,0.20,0
...,...,...,...
50903,16.962720,2545.15,22
50904,16.962633,2545.20,22
50905,16.962545,2545.25,22
50906,16.962456,2545.30,22


In [14]:
# Calculate fuel usage
# the Lexus RCF GT3 carries 100L of fuel. 
# fuel_level is 1:1 with the amount of fuel
test_lap = fuel_level[fuel_level['LapNum'] == 6]

In [15]:
test_lap

,fuel_level,ts,LapNum
13318,58.409300,665.90,6
13319,58.409300,665.95,6
13320,58.409300,666.00,6
13321,58.409300,666.05,6
13322,58.409300,666.10,6
...,...,...,...
15457,52.195305,772.85,6
15458,52.195305,772.90,6
15459,52.195305,772.95,6
15460,52.195305,773.00,6


In [ ]:
# Calculate laptime
laptime = test_lap['ts'].iloc[-1] - test_lap['ts'].iloc[0] 

# Calculate fuel used
fuel_used = test_lap['fuel_level'].iloc[0] - test_lap['fuel_level'].iloc[-1]

# 

In [27]:
print(laptime)
print(fuel_used)

107.14999999999998
6.213995000000004


In [35]:
# Create an array of the laps done in the race to iterate through a loop
num_laps = fuel_level['LapNum'].unique()

# Create new dataframe containing fuel usage
fuel_usage = pd.DataFrame(columns=['lapnum', 'laptime(s)', 'fuelused'])

for lap in num_laps:
    # isolate the lap
    timed_lap = fuel_level[fuel_level['LapNum'] == lap]
    
    # Calculate laptime
    laptime = timed_lap['ts'].iloc[-1] - timed_lap['ts'].iloc[0]
    
    # Calculate fuel used
    fuel_used = timed_lap['fuel_level'].iloc[0] - timed_lap['fuel_level'].iloc[-1]

    # save into new dataframe
    row_list = [lap, laptime, fuel_used]
    fuel_usage.loc[len(fuel_usage)] = row_list

In [36]:
fuel_usage

,lapnum,laptime(s),fuelused
0,0.0,14.60,0.000000
1,1.0,223.90,8.259010
2,2.0,107.20,6.289635
3,3.0,107.10,6.383515
4,4.0,106.90,6.292960
5,5.0,105.90,6.365580
6,6.0,107.15,6.213995
7,7.0,106.35,6.162579
8,8.0,106.25,6.290596
9,9.0,108.10,6.395820


In [37]:
fuel_usage.describe()

,lapnum,laptime(s),fuelused
count,23.00000,23.000000,23.000000
mean,11.00000,110.619565,3.262506
std,6.78233,33.387568,13.610502
min,0.00000,14.600000,-58.840571
25%,5.50000,106.300000,6.143899
50%,11.00000,107.000000,6.292960
75%,16.50000,107.650000,6.376159
max,22.00000,223.900000,8.259010
